# Notebook 02: Data Cleaning & ETL Pipeline
**Objective:** Clean all data quality issues, engineer features, and save processed dataset.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/ecommerce_dataset.csv')
print(f'Raw data: {df.shape}')
print(f'Total nulls: {df.isnull().sum().sum()}')

In [ ]:
# Step 1: Standardise city casing
# Issue: 'pune' vs 'Pune' - 200 rows with lowercase
print('Before:', df['city'].value_counts())
df['city'] = df['city'].str.strip().str.title()
print('After:', df['city'].value_counts())

In [ ]:
# Step 2: Fill missing product_name with 'Unknown'
print(f'product_name nulls: {df["product_name"].isnull().sum()}')
df['product_name'] = df['product_name'].fillna('Unknown')
print(f'After fill: {df["product_name"].isnull().sum()}')

In [ ]:
# Step 3: Fill missing payment_method with 'Unknown'
df['payment_method'] = df['payment_method'].fillna('Unknown')
print(f'Payment method value counts after fill: {df["payment_method"].value_counts()}')

In [ ]:
# Step 4: Fill missing delivery_days with median
median_days = df['delivery_days'].median()
print(f'Median delivery days: {median_days}')
df['delivery_days'] = df['delivery_days'].fillna(median_days)
print(f'Nulls remaining: {df["delivery_days"].isnull().sum()}')

In [ ]:
# Step 5: Parse dates and engineer time features
df['order_date'] = pd.to_datetime(df['order_date'])
df['month'] = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.strftime('%B')
df['week'] = df['order_date'].dt.isocalendar().week.astype(int)
print('Date columns added.')

In [ ]:
# Step 6: Engineer KPI columns
df['price_per_unit'] = (df['total_amount'] / df['quantity']).round(2)
df['is_high_value'] = (df['total_amount'] >= df['total_amount'].quantile(0.9)).astype(int)
df['fast_delivery'] = (df['delivery_days'] <= 3).astype(int)
df['high_rating'] = (df['rating'] >= 4).astype(int)
print(f'High-value orders: {df["is_high_value"].sum()}')
print(f'Fast delivery orders: {df["fast_delivery"].sum()}')

In [ ]:
# Verification
print(f'Final nulls: {df.isnull().sum().sum()}')
print(f'Final shape: {df.shape}')
df.describe()

In [ ]:
# Save cleaned data
df.to_csv('../data/processed/ecommerce_cleaned.csv', index=False)
print('Cleaned data saved to data/processed/ecommerce_cleaned.csv')